# Рекомендательная система на основе коллаборативной фильтрации

## Постановка задачи

В работе используется датасет **KION** с историей просмотров фильмов и сериалов пользователями.

**Цель работы** - исследовать, можно ли улучшить качество рекомендаций на основе коллаборативной фильтрации, если учитывать не только сам факт взаимодействия пользователя с контентом, но и дополнительную информацию о нём: глубину просмотра и давность взаимодействия.

В качестве базового варианта будет использоваться модель с бинарными взаимодействиями, где каждый просмотр имеет одинаковый вес. Далее будут рассмотрены варианты взвешивания взаимодействий по проценту просмотра, давности просмотра и их совместному учёту.

Качество рекомендаций планируется сравнивать по метрикам **MAP@10, Recall@10 и NDCG@10**.

## Этапы работы:

1. Провести первичный анализ данных: изучить структуру таблиц, типы признаков, пропуски, дубликаты, временной диапазон и распределения основных признаков.
2. Подготовить данные для моделирования и определить корректную схему временного разделения на train и test.
3. Построить простые baseline-рекомендации на основе популярности контента.
4. Построить базовую модель implicit collaborative filtering, в которой все взаимодействия имеют одинаковый вес.
5. Реализовать взвешивание взаимодействий по глубине просмотра.
6. Реализовать взвешивание взаимодействий по давности.
7. Проверить совместный учёт глубины и давности просмотра.
8. Сравнить варианты по MAP@10, Recall@10 и NDCG@10 и определить, какие изменения действительно дают прирост качества.
9. Проанализировать результаты и сформулировать итоговые выводы.

# 1. Провести первичный анализ данных: изучить структуру таблиц, типы признаков, пропуски, дубликаты, временной диапазон и распределения основных признаков

In [5]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [13]:
DATA_DIR = Path("data")

interactions_path = DATA_DIR / "interactions.csv"
users_path = DATA_DIR / "users.csv"
items_path = DATA_DIR / "items.csv"

In [14]:
interactions = pd.read_csv(
    interactions_path,
    parse_dates=["last_watch_dt"]
)

users = pd.read_csv(users_path)
items = pd.read_csv(items_path)

## 1. Первичный обзор данных

Сначала рассмотрим размер и структуру каждой таблицы, типы признаков и несколько первых наблюдений. На этом этапе важно понять, какие данные доступны для построения рекомендаций и какие признаки могут быть полезны в дальнейших экспериментах.

### 1.1 Структура таблиц
### 1.2 Пропуски и дубликаты
### 1.3 Анализ взаимодействий
### 1.4 Временная структура данных
### 1.5 Активность пользователей и объектов
### 1.6 Выводы по EDA

In [15]:
print("Interactions:", interactions.shape)
print("Users:", users.shape)
print("Items:", items.shape)

Interactions: (5476251, 5)
Users: (840197, 5)
Items: (15963, 14)


In [16]:
display(interactions.head())

,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0
3,864613,7638,2021-07-05,14483,100.0
4,964868,9506,2021-04-30,6725,100.0


In [17]:
display(users.head())

,user_id,age,income,sex,kids_flg
0,973171,age_25_34,income_60_90,М,1
1,962099,age_18_24,income_20_40,М,0
2,1047345,age_45_54,income_40_60,Ж,0
3,721985,age_45_54,income_20_40,Ж,0
4,704055,age_35_44,income_60_90,Ж,0


In [18]:
display(items.head())

,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."
2,10716,film,Тактическая сила,Tactical Force,2011.0,"криминал, зарубежные, триллеры, боевики, комедии",Канада,NaN,16.0,NaN,Адам П. Калтраро,"Адриан Холмс, Даррен Шалави, Джерри Вассерман,...",Профессиональный рестлер Стив Остин («Все или ...,"Тактическая, сила, 2011, Канада, бандиты, ганг..."
3,7868,film,45 лет,45 Years,2015.0,"драмы, зарубежные, мелодрамы",Великобритания,NaN,16.0,NaN,Эндрю Хэй,"Александра Риддлстон-Барретт, Джеральдин Джейм...","Шарлотта Рэмплинг, Том Кортни, Джеральдин Джей...","45, лет, 2015, Великобритания, брак, жизнь, лю..."
4,16268,film,Все решает мгновение,NaN,1978.0,"драмы, спорт, советские, мелодрамы",СССР,NaN,12.0,Ленфильм,Виктор Садовский,"Александр Абдулов, Александр Демьяненко, Алекс...",Расчетливая чаровница из советского кинохита «...,"Все, решает, мгновение, 1978, СССР, сильные, ж..."


In [19]:
print("INTERACTIONS")
interactions.info()

print("\nUSERS")
users.info()

print("\nITEMS")
items.info()

INTERACTIONS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5476251 entries, 0 to 5476250
Data columns (total 5 columns):
 #   Column         Dtype         
---  ------         -----         
 0   user_id        int64         
 1   item_id        int64         
 2   last_watch_dt  datetime64[ns]
 3   total_dur      int64         
 4   watched_pct    float64       
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 208.9 MB

USERS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 840197 entries, 0 to 840196
Data columns (total 5 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   user_id   840197 non-null  int64 
 1   age       826102 non-null  object
 2   income    825421 non-null  object
 3   sex       826366 non-null  object
 4   kids_flg  840197 non-null  int64 
dtypes: int64(2), object(3)
memory usage: 32.1+ MB

ITEMS
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15963 entries, 0 to 15962
Data columns (total 14 columns):
 #